# Feature Engineering — Customer Churn
Interactive companion to `src/preprocessing/feature_engineering.py`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/cleaned/telecom_customers_cleaned.csv')
df.shape

(12000, 24)

## 1. Average Monthly Spend
TotalCharges / tenure, guarding against divide-by-zero for brand new customers.

In [2]:
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges']/df['tenure'], df['MonthlyCharges'])
df[['tenure','TotalCharges','AvgMonthlySpend']].head()

,tenure,TotalCharges,AvgMonthlySpend
0,63.0,5286.72,83.916190
1,2.0,66.08,33.040000
2,1.0,59.97,59.970000
3,3.0,108.71,36.236667
4,66.0,5088.13,77.092879


## 2. Customer Lifecycle Stage
Tenure-based bucket (this dataset has no literal customer age field).

In [3]:
df['CustomerLifecycleStage'] = pd.cut(df['tenure'], bins=[-1,6,12,24,48,200],
    labels=['New (0-6mo)','Ramping (7-12mo)','Established (13-24mo)','Mature (25-48mo)','Veteran (49mo+)'])
df['CustomerLifecycleStage'].value_counts()

CustomerLifecycleStage
Mature (25-48mo)         4027
Veteran (49mo+)          3921
Established (13-24mo)    1965
New (0-6mo)              1126
Ramping (7-12mo)          961
Name: count, dtype: int64

## 3. Contract Duration (numeric) & Auto-Pay Flag

In [4]:
df['ContractDurationMonths'] = df['Contract'].map({'Month-to-month':1,'One year':12,'Two year':24})
df['IsAutoPay'] = df['PaymentMethod'].isin(['Bank transfer (automatic)','Credit card (automatic)']).astype(int)
df[['Contract','ContractDurationMonths','PaymentMethod','IsAutoPay']].head()

,Contract,ContractDurationMonths,PaymentMethod,IsAutoPay
0,Month-to-month,1,Credit card (automatic),1
1,Month-to-month,1,Electronic check,0
2,Month-to-month,1,Mailed check,0
3,Month-to-month,1,Electronic check,0
4,One year,12,Mailed check,0


## 4. Customer Value Score (0-100)
Weighted blend: 50% spend, 30% tenure, 20% contract commitment.

In [5]:
def normalize(s): return (s - s.min())/(s.max()-s.min()+1e-9)

df['CustomerValueScore'] = (
    0.5*normalize(df['MonthlyCharges'])*100 +
    0.3*normalize(df['tenure'])*100 +
    0.2*df['ContractDurationMonths'].map({1:20,12:60,24:100})
).round(2)
df['CustomerValueScore'].describe()

count    12000.000000
mean        39.167541
std         13.395345
min          4.000000
25%         29.810000
50%         39.280000
75%         48.692500
max         92.500000
Name: CustomerValueScore, dtype: float64

## 5. Service Depth & Risk Flags

In [6]:
addon_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['TotalAddOnServices'] = df[addon_cols].apply(lambda c: (c=='Yes').astype(int)).sum(axis=1)
df['HasInternetNoProtection'] = ((df['InternetService']!='No') & (df['OnlineSecurity']=='No') & (df['TechSupport']=='No')).astype(int)
df[['TotalAddOnServices','HasInternetNoProtection']].describe()

,TotalAddOnServices,HasInternetNoProtection
count,12000.000000,12000.000000
mean,1.870250,0.333583
std,1.432219,0.471512
min,0.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,3.000000,1.000000
max,6.000000,1.000000


Full 11-feature pipeline (including target encoding) is in `src/preprocessing/feature_engineering.py` — this notebook demonstrates the reasoning behind each feature for a portfolio review; the script is the source of truth used by the production pipeline.